# Run all approaches

Runs the approach notebooks in `approaches/` one after another, as if you opened each one and clicked **Run All**. Each notebook is saved **with its outputs**, so you can open it afterwards and see what happened.

> **Note:** you don't need this notebook to run a single approach. Just open that approach's notebook and run it.

> **Tip:** set `PARALLEL` above 1 to run several approaches at the same time, each in its own kernel.

## 1. Choose what to run

In [29]:
APPROACHES = "all"     # "all", or a list like ["fedasync", "fedbuff"]
PARALLEL = 1           # how many approach notebooks to run at the same time

## 2. Run them

If one notebook fails, its error is shown here and the others still run.

In [30]:
import subprocess
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import time
import nbformat
from nbclient import NotebookClient

folder = Path("approaches")
names = (sorted(p.stem for p in folder.glob("*.ipynb") if not p.stem.startswith("_"))
         if APPROACHES == "all" else list(APPROACHES))


def execute(name):
    path = folder / f"{name}.ipynb"
    nb = nbformat.read(path, as_version=4)
    t0 = time.time()
    try:
        NotebookClient(nb, timeout=None, resources={"metadata": {"path": str(folder)}}).execute(stderr=subprocess.DEVNULL)  # hide kernel start-up noise
        status = "ok"
    except Exception as e:
        status = f"FAILED: {str(e).strip().splitlines()[-1]}"
    nbformat.write(nb, path)                      # keep the outputs (and the error, if any) in the notebook
    return f"{name:24s} {status}  ({time.time() - t0:.0f}s)"


print(f"running {len(names)} approach notebook(s): {', '.join(names)}")
with ThreadPoolExecutor(PARALLEL) as ex:
    for line in ex.map(execute, names):
        print(line)
print("\nnext: open compare.ipynb")

running 5 approach notebook(s): buffered_fedavg, fedasync, fedasync_staleness, fedbuff, sync_fedavg
buffered_fedavg          ok  (12s)
fedasync                 ok  (13s)
fedasync_staleness       ok  (13s)
fedbuff                  ok  (12s)
sync_fedavg              ok  (13s)

next: open compare.ipynb
